# Vitessce Widget Tutorial

# Example usage of Neuroglancer precomputed segmentations and annotations

This notebook demonstrates `ObsSegmentationsNgPrecomputedWrapper` and `ObsPointsNgAnnotationsWrapper`, which wrap Neuroglancer precomputed segmentation/mesh data and point-annotation data (e.g. as produced by the [tissue-map-tools](https://github.com/hms-dbmi/tissue-map-tools) library) for use with the `neuroglancer` and `layerControllerBeta` views.

The `obsSets`-type file (and its corresponding view) is required for the segmentation layer to actually resolve and display any segments -- without it, segments will not be selected/colored dynamically. A static `segments` list can alternatively be passed for quick testing without a full `obsSets` pipeline -- see the alternative method below.

In [1]:
from vitessce import (
    VitessceConfig,
    CoordinationLevel as CL,
    get_initial_coordination_scope_prefix,
    ObsSegmentationsNgPrecomputedWrapper,
    CsvWrapper,
)

## 1. Configure Vitessce

In [2]:
vc = VitessceConfig(schema_version="1.0.17", name="Neuroglancer precomputed example")
dataset = vc.add_dataset("Melanoma")

# A Neuroglancer precomputed segmentation  meshes directory.
# fileUid here must match the value used below in link_views_by_dict's
# segmentationLayer coordination.
dataset.add_object(ObsSegmentationsNgPrecomputedWrapper(
    data_url="https://data-2.vitessce.io/data/sorger/melanoma_meshes",
    coordination_values={"fileUid": "segmentation"},
))

# An obsSets-type file can be used for segments to be dynamically
# selected/colored -- obsType here must match the segmentationChannel's
# obsType coordination value set below. 
dataset.add_object(CsvWrapper(
    csv_url="https://storage.googleapis.com/vitessce-demo-data/neuroglancer-march-2025/melanoma_with_embedding_filtered_ids.csv",
    data_type="obsSets",
    coordination_values={"obsType": "cell"},
    options={
        "obsIndex": "id",
        "obsSets": [{"name": "Clusters", "column": "cluster"}],
    },
))

In [3]:
ng_view = vc.add_view("neuroglancer", dataset=dataset).set_props(
    initialNgCameraState={
        'position': [49.5, 1000.5, 5209.5],
        'projectionScale': 1024,
        'projectionOrientation': [
            -0.636204183101654,
            -0.5028395652770996,
            0.5443811416625977,
            0.2145828753709793,
        ],
    },
)
lc_view = vc.add_view("layerControllerBeta", dataset=dataset)
#  TODO: until support to load the segments is added in NG-View
# The obsSets view is not required for the segmentation to load, but
# a mounted obsSets view is needed to trigger the underlying data hook 
# that resolves obsSets data for the segmentation channel.
obs_sets_view = vc.add_view("obsSets", dataset=dataset)
vc.layout(ng_view | (lc_view / obs_sets_view))

## 2. Coordinate the views

Two separate `link_views_by_dict` calls are needed:
- A plain (non-meta) link for shared spatial rendering mode and camera position/rotation.
- A multi-level (meta) link for the segmentation layer + channel, mirroring the shape
  `obsSegmentations.ng-precomputed` files require to resolve correctly.

In [4]:
vc.link_views_by_dict([ng_view, lc_view], {
    "spatialRenderingMode": "3D",
    "spatialZoom": 0,
    "spatialTargetX": 0,
    "spatialTargetY": 0,
    "spatialTargetZ": 0,
    "spatialRotationX": 0,
    "spatialRotationY": 0,
    "spatialRotationOrbit": 0,
}, meta=False)

vc.link_views_by_dict([ng_view, lc_view], {
    "segmentationLayer": CL([{
        "fileUid": "segmentation",
        "spatialLayerOpacity": 1,
        "spatialTargetResolution": None,
        "spatialLayerVisible": True,
        "segmentationChannel": CL([{"obsType": "cell", "spatialChannelVisible": True}]),
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix("A", "obsSegmentations"))

## 3. Create the Vitessce widget

In [5]:
vw = vc.widget(custom_js_url="http://localhost:9000/packages/main/dev/dist/index.js")
vw

## 4. Alternate way to load segments: providing an explicit array

Instead of pointing at a remote `obsSets` CSV (as in section 1 above), you can select and color a specific, known set of segments directly from a Python list/dict. Native Neuroglancer itself supports specifying segments this way, as a plain array ("segments": [...]) in its own JSON state — this section replicates that same capability through Vitessce's own coordination system.
This adds two files instead of one `obsSets.csv`
- `obsFeatureMatrix.csv` -- just the segment IDs, defining which observations exist.
- `obsColors.csv` -- an explicit `id -> color` mapping. -- optional

The `segmentationChannel` coordination also changes slightly: `obsColorEncoding` is set to `'obsColors'` (using the explicit colors above) instead of relying on cluster-based coloring.

In [6]:
from vitessce import make_ids_csv_data_url, make_colors_csv_data_url

segment_ids = [612, 3351, 4328, 6531, 8446]
segment_colors = {
    612: '#d74242',
    3351: '#b9d742',
    4328: '#42d77d',
    6531: '#427dd7',
    8446: '#b942d7',
}

vc_alt = VitessceConfig(
    schema_version='1.0.17',
    name='Neuroglancer precomputed example (explicit segments)',
)
dataset_alt = vc_alt.add_dataset('Melanoma')

dataset_alt.add_object(ObsSegmentationsNgPrecomputedWrapper(
    data_url='https://data-2.vitessce.io/data/sorger/melanoma_meshes',
    coordination_values={'fileUid': 'segmentation'},
))

# IDs only -- defines which observations exist for this obsType.
dataset_alt.add_object(CsvWrapper(
    csv_url=make_ids_csv_data_url(segment_ids),
    data_type='obsFeatureMatrix',
    coordination_values={
        'obsType': 'cell', 'featureType': 'feature', 'featureValueType': 'value',
    },
))

# Explicit id -> color mapping.
dataset_alt.add_object(CsvWrapper(
    csv_url=make_colors_csv_data_url(segment_colors),
    data_type='obsColors',
    options={'obsIndex': 'id', 'obsColors': 'color'},
    coordination_values={'obsType': 'cell'},
))

ng_view_alt = vc_alt.add_view('neuroglancer', dataset=dataset_alt).set_props(
    initialNgCameraState={
        'position': [49.5, 1000.5, 5209.5],
        'projectionScale': 1024,
        'projectionOrientation': [
            -0.636204183101654,
            -0.5028395652770996,
            0.5443811416625977,
            0.2145828753709793,
        ],
    },
)
lc_view_alt = vc_alt.add_view('layerControllerBeta', dataset=dataset_alt)
vc_alt.layout(ng_view_alt | lc_view_alt)

vc_alt.link_views_by_dict([ng_view_alt, lc_view_alt], {
    'spatialRenderingMode': '3D',
    'spatialZoom': 0, 'spatialTargetX': 0, 'spatialTargetY': 0, 'spatialTargetZ': 0,
    'spatialRotationX': 0, 'spatialRotationY': 0, 'spatialRotationOrbit': 0,
}, meta=False)

vc_alt.link_views_by_dict([ng_view_alt, lc_view_alt], {
    'segmentationLayer': CL([{
        'fileUid': 'segmentation',
        'spatialLayerOpacity': 1,
        'spatialTargetResolution': None,
        'spatialLayerVisible': True,
        'segmentationChannel': CL([{
            'obsType': 'cell',
            'featureType': 'feature',
            'featureValueType': 'value',
            'spatialChannelVisible': True,
            'obsColorEncoding': 'obsColors',
        }]),
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix('A', 'obsSegmentations'))

# TODO: drop the custom_js_url when updates released
vw_alt = vc_alt.widget(custom_js_url='http://localhost:9000/packages/main/dev/dist/index.js')
vw_alt